# 04 · Inventory Intelligence: EDA and Preprocessing

**Dataset:** `data/raw/inventory/train.csv` (15,243 rows, one row per SKU).

**Output:** `data/cleaned/inventory/inventory_cleaned.csv`, read by `src/models/train_inventory.py` and the Inventory page.

**Target:** `went_on_backorder` (1 = the item went on backorder).

**Important:** the source file has a `Validation` column (0/1). It is a split flag from the original data, not a business signal, so it is **dropped**. `sku` is an identifier and is kept in the cleaned file but **excluded** from the model inputs.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

# Works whether Jupyter is opened from the project root or from the notebooks/ folder.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
print("Project root:", ROOT)

## 1. Load and inspect

In [ ]:
RAW_PATH = ROOT / "data" / "raw" / "inventory" / "train.csv"
CLEAN_PATH = ROOT / "data" / "cleaned" / "inventory" / "inventory_cleaned.csv"

df_raw = pd.read_csv(RAW_PATH)
print("Shape:", df_raw.shape)
print("Columns:", list(df_raw.columns))
df_raw.info()
display(df_raw.head())

## 2. Missing values, duplicates and target balance

In [ ]:
missing = df_raw.isna().sum()
print("Columns with missing values:")
print(missing[missing > 0])
print("\nExact duplicate rows:", df_raw.duplicated().sum())
print("Duplicate sku values:", df_raw["sku"].duplicated().sum())
print("\nTarget values:", sorted(df_raw["went_on_backorder"].dropna().unique()))
print("Missing target:", df_raw["went_on_backorder"].isna().sum())
print("\nClass balance (share of rows):")
print(df_raw["went_on_backorder"].value_counts(normalize=True).round(4))

## 3. Class imbalance

Backorders are much rarer than normal stock, so a model that always predicts "no backorder" looks accurate but misses every backorder. This is why the project reports **recall** and **ROC-AUC**, and why the classifier uses `class_weight="balanced"` **after** the train/test split. No balancing (such as SMOTE) is applied to the test set.

In [ ]:
counts = df_raw["went_on_backorder"].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(["No backorder (0)", "Backorder (1)"], counts.values, color=["#3B8F7B", "#A7F432"])
ax.set_title("Class balance: went_on_backorder")
ax.set_ylabel("Number of SKUs")
for i, v in enumerate(counts.values):
    ax.text(i, v, f"{v:,}", ha="center", va="bottom")
plt.tight_layout()
plt.show()

## 4. Numerical validation

Negative stock, sales or forecasts would be impossible. Missing `lead_time` values are kept as missing. The training pipeline fills them with the median **inside the training split only**, which avoids leaking test information.

In [ ]:
num_cols = [c for c in df_raw.columns if c not in ("sku", "went_on_backorder", "Validation")
            and pd.api.types.is_numeric_dtype(df_raw[c])]
display(df_raw[num_cols].describe().T.round(2))
neg = {c: int((df_raw[c] < 0).sum()) for c in num_cols if (df_raw[c] < 0).any()}
print("Columns with negative values:", neg or "none")

## 5. Cleaning decisions

1. **Drop `Validation`**, the original split flag, which is not a model input.
   Also convert the **-99 sentinel** in `perf_6_month_avg` and `perf_12_month_avg` to missing. Those 2,089 cells are "no history" codes, not real performance values.
2. **Remove rows with a missing or invalid target**, because a supervised model cannot learn from them.
3. **Remove exact duplicate rows** (none expected, but the check is kept for future data).
4. **Keep missing feature values** (for example `lead_time`). They are imputed later, inside the pipeline.

In [ ]:
df = df_raw.drop(columns=["Validation"], errors="ignore").copy()

# -99 is a sentinel code for "no performance history", not a real measurement.
# Leaving it in would make the model read -99 as a very poor performance value.
sentinel_cols = [c for c in df.columns if (pd.to_numeric(df[c], errors="coerce") == -99).any()]
print("Columns with -99 sentinel:", sentinel_cols)
print("Sentinel cells before cleaning:", int((df[sentinel_cols] == -99).sum().sum()))
df[sentinel_cols] = df[sentinel_cols].replace(-99, np.nan)
print("Sentinel cells after cleaning (now missing):", int(df[sentinel_cols].isna().sum().sum()))

before = len(df)
df = df[df["went_on_backorder"].isin([0, 1])]
df = df.drop_duplicates()
print("Rows removed:", before - len(df))
print("Rows after cleaning:", len(df), "| columns:", df.shape[1])

## 6. Save the cleaned dataset

In [ ]:
CLEAN_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(CLEAN_PATH, index=False)
print("Saved cleaned dataset:", CLEAN_PATH)

## 7. EDA: which signals differ between backordered and normal SKUs?

In [ ]:
signals = ["lead_time", "national_inv", "forecast_3_month", "sales_1_month", "perf_6_month_avg"]
comparison = df.groupby("went_on_backorder")[signals].median().T
comparison.columns = ["no_backorder_median", "backorder_median"]
display(comparison.round(3))